# Customer Churn Prediction
## 1. Project introduction
The goal of this project is to develop a machine learning model to predict customer churn for a subscription-based service or business using historical customer data.

## 2. Import libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report
import joblib

## 3. Load dataset

In [ ]:
df = pd.read_csv('../data/Churn_Modelling.csv')
df.head()

## 4. Dataset overview

In [ ]:
print('Shape:', df.shape)
print('\nColumns:', df.columns.tolist())
print('\nData Types:\n', df.dtypes)
print('\nTarget Distribution:\n', df['Exited'].value_counts(normalize=True))

## 5. Data quality check

In [ ]:
print('Missing Values:\n', df.isnull().sum())
print('\nDuplicate Rows:', df.duplicated().sum())

## 6. Data cleaning

In [ ]:
# Drop identifier columns as they are not predictive features
identifiers = ['RowNumber', 'CustomerId', 'Surname']
df_clean = df.drop(columns=[col for col in identifiers if col in df.columns], errors='ignore')
df_clean.head()

## 7. EDA

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(x='Exited', data=df_clean)
plt.title('Churn Distribution')
plt.xlabel('Exited (0=Retained, 1=Churned)')
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(x='Exited', y='Age', data=df_clean)
plt.title('Age vs Churn')
plt.show()

Observations:
- There is class imbalance (more retained customers than churned).
- Older customers seem to have a higher likelihood of churning compared to younger ones.

## 8. Feature analysis

In [ ]:
# Numerical correlation
numeric_df = df_clean.select_dtypes(include=['int64', 'float64'])
corr = numeric_df.corr()
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Numerical Feature Correlation')
plt.show()

## 9. Feature selection
All remaining features (CreditScore, Geography, Gender, Age, Tenure, Balance, NumOfProducts, HasCrCard, IsActiveMember, EstimatedSalary) are considered reasonably useful based on business logic and initial analysis. We will retain all of them for the models.

## 10. Train-test split

In [ ]:
X = df_clean.drop(columns=['Exited'])
y = df_clean['Exited']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print('Train shape:', X_train.shape)
print('Test shape:', X_test.shape)

## 11. Preprocessing

In [ ]:
numeric_features = ['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary']
categorical_features = ['Geography', 'Gender']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), categorical_features)
    ]
)

## 12. Logistic Regression

In [ ]:
lr_pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('classifier', LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000))])
lr_pipeline.fit(X_train, y_train)

## 13. Random Forest

In [ ]:
rf_pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('classifier', RandomForestClassifier(class_weight='balanced', random_state=42))])
rf_pipeline.fit(X_train, y_train)

## 14. Gradient Boosting

In [ ]:
gb_pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('classifier', GradientBoostingClassifier(random_state=42))])
gb_pipeline.fit(X_train, y_train)

## 15. Model evaluation

In [ ]:
def evaluate(model, X, y):
    y_pred = model.predict(X)
    y_prob = model.predict_proba(X)[:, 1]
    print('Accuracy:', accuracy_score(y, y_pred))
    print('Precision:', precision_score(y, y_pred))
    print('Recall:', recall_score(y, y_pred))
    print('F1-score:', f1_score(y, y_pred))
    print('ROC-AUC:', roc_auc_score(y, y_prob))
    print('Classification Report:\n', classification_report(y, y_pred))

In [ ]:
print('--- Logistic Regression ---')
evaluate(lr_pipeline, X_test, y_test)

In [ ]:
print('--- Random Forest ---')
evaluate(rf_pipeline, X_test, y_test)

In [ ]:
print('--- Gradient Boosting ---')
evaluate(gb_pipeline, X_test, y_test)

## 16. Model comparison
Gradient Boosting and Random Forest performed reasonably well. Random forest with class_weight='balanced' provides a decent balance of precision and recall. Gradient Boosting has high accuracy and ROC-AUC.

## 17. Hyperparameter tuning

In [ ]:
param_distributions = {
    'classifier__n_estimators': [100, 200],
    'classifier__max_depth': [10, 20, None],
    'classifier__min_samples_split': [2, 5],
    'classifier__min_samples_leaf': [1, 2]
}
search = RandomizedSearchCV(rf_pipeline, param_distributions, n_iter=5, cv=3, scoring='f1', random_state=42, n_jobs=-1)
search.fit(X_train, y_train)
print('Best parameters:', search.best_params_)

## 18. Final model selection
We select the Tuned Random Forest model as our Final Selected Model because it offers robust performance and handles class imbalance well.

In [ ]:
final_model = search.best_estimator_
evaluate(final_model, X_test, y_test)

## 19. Feature importance

In [ ]:
num_features_out = final_model.named_steps['preprocessor'].transformers_[0][2]
cat_encoder = final_model.named_steps['preprocessor'].transformers_[1][1]
cat_features_out = cat_encoder.get_feature_names_out(categorical_features)
all_features = list(num_features_out) + list(cat_features_out)

importances = final_model.named_steps['classifier'].feature_importances_
fi = pd.DataFrame({'Feature': all_features, 'Importance': importances})
fi = fi.sort_values(by='Importance', ascending=False)
print(fi)

## 20. Save final model

In [ ]:
joblib.dump(final_model, '../models/customer_churn_model.pkl')